In [ ]:
pip install google-adk

In [ ]:
from google.adk.models import LlmRequest, LlmResponse
from google.adk.agents.callback_context import CallbackContext
from google.genai import types

In [ ]:
from google.adk.tools import google_search, agent_tool
from google.adk.agents import LlmAgent, SequentialAgent

SEARCH_INSTRUCTIONS = """
You are the Search Agent. Your sole purpose is to gather raw information to answer the user's question.

Your tasks:
1. Analyze the user's question and identify the core intent and keywords.
2. Use the provided google_search tool to find the most accurate and up-to-date data.
3. Extract three specific components: the Answer, the Title of the page, and the Source URL.
4. Present this data objectively. Do not attempt to format it into a final response; simply provide the facts and the metadata (Title/Link) clearly.
"""

search_agent = LlmAgent(
    model="gemini-2.5-flash-lite",
    name="search_agent",
    description="",
    instruction=SEARCH_INSTRUCTIONS,
    tools=[google_search]
)

In [ ]:
CRITIQUE_INSTRUCTIONS = """
You are the Critique Agent. Your role is to act as a rigorous editor for the Search Agent's output.

Your tasks:
1. Evaluate the Search Agent's response against the user's original question. Does it actually answer the question?
2. Check for technical requirements: Is there a Title? Is there a valid Link? Is the information clear?
3. Look for "hallucinations" or logical inconsistencies.
4. Provide a list of specific, actionable suggestions for improvement. If the answer is perfect, state that no changes are needed.
5. Focus on improving clarity, tone, and the transparency of the source.
"""

critique_agent = LlmAgent(
    model="gemini-2.5-flash-lite",
    name="critique_agent",
    description="Checks that the data retrieve from search is sufficient",
    instruction=CRITIQUE_INSTRUCTIONS
)

In [ ]:
REFINE_INSTRUCTIONS = """
You are the Refine Agent. Your goal is to synthesize the Search Agent's data and the Critique Agent's feedback into a perfect final response.

Your tasks:
1. Rewrite the initial findings into a professional, engaging, and easy-to-read answer.
2. Address every suggestion made by the Critique Agent to ensure the highest quality.
3. You MUST format the final output as follows:
   - Provide the answer in one or two concise paragraphs.
   - Below the answer, include a section titled "Source".
   - Under "Source", list the Page Title and the clickable URL Link.
4. Ensure the tone is helpful and that the link is clearly visible.
"""

refine_agent = LlmAgent(
    model="gemini-2.5-flash-lite",
    name="refine_agent",
    description="Refines the output of the Search Agent",
    instruction=REFINE_INSTRUCTIONS
)

In [ ]:
greeter_agent = LlmAgent(
    model="gemini-2.5-flash-lite",
    name="greeter_agent",
    description="Greets the user and asks for a question",
    instruction="""
    You are the Greeter Agent. Your sole purpose is to greet the user and ask them a question
    to begin the conversation."""
)

In [ ]:
root_agent = SequentialAgent(
    name="root_agent",
    sub_agents=[greeter_agent, search_agent, critique_agent, refine_agent]
)

In [ ]:
from vertexai.preview import reasoning_engines

app = reasoning_engines.AdkApp(
    agent=root_agent
)

user_id = "test-user-id"
session = app.create_session(user_id=user_id)

/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:825: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


In [ ]:
from IPython.display import Markdown, display

for event in app.stream_query(
  user_id=user_id,
  session_id=session.id,
  message="Why is the sky blue?",
):
    # Check if the event has content and parts
    if "content" in event and "parts" in event["content"]:
        for part in event["content"]["parts"]:
            # Only try to display if the part actually contains text
            if "text" in part:
                display(Markdown(part["text"]))
            # Optional: Print a status when a tool is being called
            elif "function_call" in part:
                print(f"Tool Calling: {part['function_call']['name']}...")

Hello there! I'm happy to help. To start, could you tell me what you're curious about today?

The sky appears blue due to a phenomenon called Rayleigh scattering. When sunlight enters Earth's atmosphere, it collides with tiny gas molecules, primarily nitrogen and oxygen. These molecules scatter shorter wavelengths of light, such as blue and violet, more effectively than longer wavelengths like red. Although violet light is scattered even more than blue light, our eyes are more sensitive to blue, and there is less violet light in sunlight to begin with, which is why we perceive the sky as blue. This scattering effect causes blue light to be re-emitted in all directions, filling the sky with its color.

Here's a critique of the Search Agent's response:

**Critique:**

The Search Agent's response provides a good, scientifically accurate explanation for why the sky is blue. It correctly identifies Rayleigh scattering as the primary reason and explains the scattering of shorter wavelengths (blue and violet) more effectively than longer ones. It also addresses the nuance of violet light scattering more but our perception being blue due to eye sensitivity and initial light composition.

However, there are a few areas for improvement:

*   **Lack of Title:** The response is missing a clear title that would immediately tell the user what the information is about.
*   **No Link:** While the information is factual, there is no source link provided to allow the user to verify or explore the information further. This is a crucial element for transparency and credibility.
*   **Slightly Dense:** While accurate, the explanation could be made slightly more accessible with clearer sentence structure in a few places. For instance, the sentence about violet light could be broken down or rephrased for absolute clarity.

**Suggestions for Improvement:**

1.  **Add a Title:** Include a concise title at the beginning, such as "Why the Sky Appears Blue" or "The Science Behind the Blue Sky."
2.  **Provide a Source Link:** Include a link to a reputable source (e.g., NASA, NOAA, a well-known science education website) that elaborates on Rayleigh scattering and the color of the sky.
3.  **Refine Sentence Structure (Minor):** Consider slightly rephrasing the sentence: "Although violet light is scattered even more than blue light, our eyes are more sensitive to blue, and there is less violet light in sunlight to begin with, which is why we perceive the sky as blue." This could be broken into two sentences for easier digestion. For example: "While violet light is scattered even more intensely than blue light, our eyes are more sensitive to blue. Additionally, there's slightly less violet light in sunlight to begin with. These factors combine to make us perceive the sky as blue." (This is a minor suggestion, as the original is still understandable).

The sky appears blue due to a phenomenon called Rayleigh scattering. When sunlight enters Earth's atmosphere, it interacts with tiny gas molecules. These molecules scatter shorter wavelengths of light, like blue and violet, more effectively than longer wavelengths, such as red. Although violet light is scattered even more intensely than blue light, our eyes are more sensitive to blue, and there's slightly less violet light in sunlight to begin with. These factors combine to make us perceive the sky as blue, as this scattered blue light is re-emitted in all directions, filling the sky.

Source
Why the Sky Appears Blue
https://scied.ucar.edu/learning-zone/how-weather-works/why-sky-blue